
# AgriSense — Autonomous Crop Advisory Agent (Kaggle Submission Notebook)

**Track:** Agents for Good  
**Goal:** Demo a multi-agent pipeline (Diagnosis → Advisory → Market → Climate) built with ADK-style patterns and ready for Kaggle submission.

This notebook:
- Creates the project file structure (agents/, tools/, utils/) with starter code.
- Shows how to run the orchestrator end-to-end (with placeholders).
- Explains where to paste your model & Gemini code from the 5-Day notebooks.
- Includes the writeup content ready for the Kaggle writeup (replace placeholders as needed).



## Environment & Requirements

Install required Python packages (optional). On Kaggle, some packages may already be available.

```bash
!pip install -q numpy pillow requests
```

> **Important:** Do **not** put API keys in the notebook. Use Kaggle Secrets or environment variables:
- `GEMINI_API_KEY`
- `MCP_GATEWAY_URL`
- `WEATHER_API_KEY`


In [ ]:
import os
BASE='/mnt/data/agrisense'
for rel,content in {'agents/orchestrator.py': '# orchestrator.py\nfrom typing import Dict, Any\nfrom google.adk.sessions import InMemorySessionService\nfrom tools.mcp_tool import MCPToolClient\nfrom utils.memory_bank import MemoryBank\nfrom utils.logging_tracing import log_event\nfrom agents.diagnosis_agent import DiagnosisAgent\nfrom agents.advisory_agent import AdvisoryAgent\nfrom agents.market_agent import MarketAgent\nfrom agents.climate_agent import ClimateAgent\n\nclass Orchestrator:\n    def __init__(self, *, session_service=None, mcp_gateway_url=None, gemini_client=None, local_model=None):\n        self.session_service = session_service or InMemorySessionService()\n        self.memory = MemoryBank()\n        self.mcp = MCPToolClient(gateway_url=mcp_gateway_url)\n        self.gemini = gemini_client\n        self.local_model = local_model\n        self.diagnosis = DiagnosisAgent(mcp_tool=self.mcp, gemini_client=self.gemini, local_model=self.local_model)\n        self.advisory = AdvisoryAgent(gemini_client=self.gemini)\n        self.market = MarketAgent(market_tool=None)\n        self.climate = ClimateAgent(weather_tool=None)\n\n    def run_case(self, user_id: str, image_input, crop_type: str = None) -> Dict[str, Any]:\n        log_event(f"orchestrator.start user={user_id}")\n        session = self.session_service.start_session(user_id)\n        user_memory = self.memory.get_user(user_id) or {}\n        diag = self.diagnosis.run(image_input, crop_type=crop_type)\n        advice = self.advisory.run({\'diagnosis\': diag, \'user_memory\': user_memory, \'crop\': crop_type})\n        try:\n            market_info = self.market.get_market_insights(crop_type or diag.get(\'crop\') or "unknown")\n        except Exception as e:\n            market_info = {"error": str(e)}\n        try:\n            climate_info = self.climate.assess_location_risk(location=user_memory.get("location"))\n        except Exception as e:\n            climate_info = {"error": str(e)}\n        record = {"last_diag": diag, "last_advice": advice, "market": market_info, "climate": climate_info}\n        user_memory.update(record)\n        self.memory.save_user(user_id, user_memory)\n        log_event(f"orchestrator.end user={user_id}")\n        return {"diagnosis": diag, "advice": advice, "market": market_info, "climate": climate_info}\n', 'agents/diagnosis_agent.py': "# diagnosis_agent.py\nimport io, base64, numpy as np\nfrom PIL import Image\n\nclass DiagnosisAgent:\n    def __init__(self, mcp_tool=None, gemini_client=None, local_model=None, backend_preference=None, class_names=None):\n        self.mcp = mcp_tool\n        self.gemini = gemini_client\n        self.local_model = local_model\n        self.backends = backend_preference or ['local','mcp','gemini']\n        self.class_names = class_names\n\n    def _load_image(self, image_input):\n        if isinstance(image_input, (bytes, bytearray)):\n            return Image.open(io.BytesIO(image_input)).convert('RGB')\n        if hasattr(image_input, 'read'):\n            return Image.open(image_input).convert('RGB')\n        if isinstance(image_input, str):\n            return Image.open(image_input).convert('RGB')\n        raise ValueError('Unsupported image input')\n\n    def _preprocess(self, pil_img, target=(224,224)):\n        pil_img = pil_img.resize(target)\n        arr = np.array(pil_img).astype('float32')/255.0\n        return arr\n\n    def _run_local(self, pil_img):\n        raise NotImplementedError('Replace with your local model inference from notebooks')\n\n    def _run_mcp(self, pil_img):\n        if self.mcp is None:\n            raise RuntimeError('MCP backend selected but no mcp client provided')\n        buf = io.BytesIO()\n        pil_img.save(buf, format='JPEG')\n        b64 = base64.b64encode(buf.getvalue()).decode('utf-8')\n        payload = {'image_base64': b64}\n        result = self.mcp.call(tool_name='classify_image', payload=payload)\n        return self._format(result)\n\n    def _run_gemini(self, pil_img):\n        raise NotImplementedError('Replace with your Gemini client inference code')\n\n    def _format(self, raw):\n        if isinstance(raw, dict) and 'label' in raw and 'confidence' in raw:\n            return raw\n        return {'label':'unknown','confidence':0.0,'raw':str(raw)}\n\n    def run(self, image_input, crop_type=None):\n        pil = self._load_image(image_input)\n        last_exc = None\n        for backend in self.backends:\n            try:\n                if backend.startswith('local'):\n                    res = self._run_local(pil)\n                elif backend.startswith('mcp'):\n                    res = self._run_mcp(pil)\n                else:\n                    res = self._run_gemini(pil)\n                res.setdefault('crop', crop_type)\n                return res\n            except Exception as e:\n                last_exc = e\n                continue\n        raise RuntimeError('All inference backends failed') from last_exc\n", 'agents/advisory_agent.py': '# advisory_agent.py\nclass AdvisoryAgent:\n    def __init__(self, gemini_client=None, language=\'en\'):\n        self.gemini = gemini_client\n        self.language = language\n\n    def _prompt(self, diagnosis, user_memory, crop):\n        return f"""You are an expert agricultural advisor.\nCrop: {crop}\nDiagnosis: {diagnosis.get(\'label\')} (confidence: {diagnosis.get(\'confidence\')})\nProvide clear, actionable steps for a smallholder farmer: immediate actions, treatment (name & dosage), safety, organic alternatives, and follow-up timing."""\n\n    def _call_gemini(self, prompt):\n        raise NotImplementedError(\'Paste your Gemini call here\')\n\n    def _fallback(self, diagnosis):\n        label = diagnosis.get(\'label\',\'unknown\')\n        mapping = {\n            \'leaf_blight\': \'Remove affected leaves; apply fungicide as per local label; monitor after 7 days.\',\n            \'healthy\': \'No action required.\',\n            \'unknown\': \'Collect a clearer image and re-upload.\'\n        }\n        return mapping.get(label, mapping[\'unknown\'])\n\n    def run(self, payload):\n        diagnosis = payload.get(\'diagnosis\', {})\n        user_memory = payload.get(\'user_memory\', {})\n        crop = payload.get(\'crop\', \'unknown\')\n        if self.gemini:\n            try:\n                return self._call_gemini(self._prompt(diagnosis, user_memory, crop))\n            except Exception:\n                pass\n        return self._fallback(diagnosis)\n', 'agents/market_agent.py': "# market_agent.py\nclass MarketAgent:\n    def __init__(self, market_tool=None):\n        self.market_tool = market_tool\n\n    def get_market_insights(self, crop, region=None):\n        if self.market_tool:\n            return self.market_tool.get_price(crop, region)\n        return {'crop':crop,'region':region,'avg_price_per_kg':None,'trend':'unknown'}\n", 'agents/climate_agent.py': "# climate_agent.py\nclass ClimateAgent:\n    def __init__(self, weather_tool=None):\n        self.weather_tool = weather_tool\n\n    def assess_location_risk(self, location):\n        if self.weather_tool and location:\n            return self.weather_tool.get_forecast_risk(location)\n        return {'location':location,'risk':'unknown'}\n", 'tools/mcp_tool.py': "# mcp_tool.py\nimport os, requests\nclass MCPToolClient:\n    def __init__(self, gateway_url=None):\n        self.gateway = gateway_url or os.environ.get('MCP_GATEWAY_URL')\n    def call(self, tool_name, payload):\n        if self.gateway:\n            resp = requests.post(self.gateway, json={'tool':tool_name,'payload':payload}, timeout=15)\n            resp.raise_for_status()\n            return resp.json()\n        raise RuntimeError('No MCP gateway configured')\n", 'tools/weather_tool.py': "# weather_tool.py\nimport os, requests\nclass WeatherTool:\n    def __init__(self, api_key=None):\n        self.api_key = api_key or os.environ.get('WEATHER_API_KEY')\n        self.base = os.environ.get('WEATHER_API_BASE','https://api.open-meteo.com/v1/forecast')\n    def get_forecast_risk(self, location, days=3):\n        try:\n            lat,lon = map(float, location.split(','))\n        except:\n            return {'location':location,'risk':'unknown','note':'Provide lat,lon'}\n        params = {'latitude':lat,'longitude':lon,'hourly':'temperature_2m,precipitation','forecast_days':days}\n        resp = requests.get(self.base, params=params, timeout=10)\n        resp.raise_for_status()\n        return {'location':location,'forecast':resp.json()}\n", 'tools/market_tool.py': "# market_tool.py\nclass MarketTool:\n    def __init__(self, api_config=None):\n        self.api_config = api_config or {}\n    def get_price(self, crop, region=None):\n        return {'crop':crop,'region':region,'avg_price_per_kg':None,'trend':'unknown'}\n", 'utils/memory_bank.py': '# memory_bank.py\nimport threading\nclass MemoryBank:\n    def __init__(self):\n        self._lock = threading.Lock()\n        self.users = {}\n    def save_user(self, user_id, data):\n        with self._lock:\n            self.users[user_id] = data.copy()\n    def get_user(self, user_id):\n        with self._lock:\n            return self.users.get(user_id)\n', 'utils/logging_tracing.py': "# logging_tracing.py\nimport logging, time, json\nlogger = logging.getLogger('agrisense')\nlogger.setLevel(logging.INFO)\nhandler = logging.StreamHandler()\nif not logger.handlers:\n    logger.addHandler(handler)\ndef log_event(message, **meta):\n    data = {'ts': time.time(), 'message': message}\n    data.update(meta)\n    logger.info(json.dumps(data))\n", 'README.md': '# AgriSense — Starter\n\nThis directory contains starter code for the AgriSense capstone. Replace NOT_IMPLEMENTED placeholders with your notebook-specific code:\n- Local model inference\n- Gemini client calls\n- MCP server gateway (optional)\n\nDo not commit API keys. Use Kaggle Secrets or environment variables.\n'}.items():
    path = os.path.join(BASE, rel)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path,'w',encoding='utf-8') as f:
        f.write(content)
print('Wrote project files to /mnt/data/agrisense')


In [ ]:
# Demo: instantiate orchestrator and run with a placeholder image
from agents.orchestrator import Orchestrator
from PIL import Image
img_path = '/mnt/data/agrisense/demo_leaf.jpg'
Image.new('RGB', (224,224), color=(100,200,100)).save(img_path)
orch = Orchestrator(mcp_gateway_url=None, gemini_client=None, local_model=None)
try:
    out = orch.run_case(user_id='demo_farmer', image_input=img_path, crop_type='wheat')
    print('Orchestrator output (demo):')
    print(out)
except Exception as e:
    print('Note: some functions are placeholders and will raise NotImplementedError or RuntimeError until you paste your notebook code.')
    print('Error:', e)



## Next steps — paste your notebook code

Replace the `NotImplementedError` placeholders in:
- `agents/diagnosis_agent.py::_run_local`  → paste your model loading & inference code
- `agents/diagnosis_agent.py::_run_gemini` → paste your Gemini Vision call
- `agents/advisory_agent.py::_call_gemini` → paste your LLM prompt + call

If you used an MCP server via `npx @modelcontextprotocol/server-everything`, either:
- Start the MCP server and set `MCP_GATEWAY_URL` env var to the gateway, or
- Pass the `McpToolset` object from your notebook into `Orchestrator` instead of MCPToolClient.

Finally, test again by running the demo cell above.


## Submission Writeup (paste into Kaggle writeup form)

### Title
AgriSense — Autonomous Crop Advisory & Disease Prediction Agent

### Problem
Smallholder farmers face delayed detection of crop diseases, lack personalized advisory, and have limited access to real-time market and climate guidance. These issues reduce yields and increase vulnerability.

### Solution
AgriSense is a multi-agent system combining image-based diagnosis, LLM-powered advisory, market insights, and climate risk monitoring. The system uses a modular architecture where specialized agents handle diagnosis, advisory, market, and climate tasks. Memory and session services store farmer history for personalization.

### Architecture
- **Orchestrator:** Coordinates agent calls and persists session data.
- **DiagnosisAgent:** Multi-backend classifier (local/MCP/Gemini Vision).
- **AdvisoryAgent:** LLM-guided actionable instructions with template fallback.
- **MarketAgent & ClimateAgent:** Provide market pricing and weather-based risk signals.
- **Tools:** MCP client, weather tool, market tool.
- **Memory Bank:** In-memory user data for personalization.

### Implementation notes
- Demonstrates multi-agent orchestration, custom tools, sessions & memory, long-running climate monitoring, and observability.
- Replace placeholder inference & Gemini calls with code from the 5-day notebooks to match the ADK patterns taught in the course.
- For bonus points, deploy an Agent Engine runtime or record a short 2–3 minute demo showing image → diagnosis → advice flow.

### Evaluation & Metrics
- Evaluate classification accuracy using labeled test images; report precision/recall per disease class.
- Compare latency between backends (local vs MCP vs Gemini).
- Show sample user flows and edge-case handling.

(Extend this to ~1500 words for the Kaggle writeup, and include images/diagrams as needed.)
